# H&M 2년 N 조건부 후보상품 관계 진단
기존 seed-42 M1에서 Dunnhumby와 동일하게 상품군 전이의 N 귀속을 pooled·actual N·degree-matched shuffled N으로 비교합니다. 학습·재정렬·최종 test·holdout은 수행하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '636d4ba'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip().startswith(REVIEWED_SHA)

In [ ]:
import importlib
import json
import torch
import clv_m3_clv_conditioned_category_transition_graph as transition_graph
transition_graph = importlib.reload(transition_graph)
import lightgcn_clv_candidate_relation_diagnostic as candidate_base
candidate_base = importlib.reload(candidate_base)
import lightgcn_clv_n_conditioned_candidate_relation_diagnostic as n_diagnostic
n_diagnostic = importlib.reload(n_diagnostic)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert n_diagnostic.CODE_VERSION == 'm1-n-conditioned-candidate-relation-diagnostic-v1'
cfg = n_diagnostic.configure_n_conditioned_candidate_relation_diagnostic('hm')
print(json.dumps(n_diagnostic.preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
transition_graph = importlib.reload(transition_graph)
candidate_base = importlib.reload(candidate_base)
n_diagnostic = importlib.reload(n_diagnostic)
cfg = n_diagnostic.configure_n_conditioned_candidate_relation_diagnostic('hm')
paths = n_diagnostic.run_n_conditioned_candidate_relation_diagnostic(cfg)

In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv(paths['relation_summary_csv'])
focus = summary.group_type.isin(['overall', 'fixed_clv_segment', 'high_clv_composition'])
columns = [
    'arm', 'group_type', 'group', 'n_users', 'candidate_pair_count',
    'pair_balanced_win_rate', 'macro_user_balanced_win_rate',
    'pair_strict_win_rate', 'pair_tie_rate', 'mean_pair_score_difference',
]
print('1) pooled / 실제 N 조건부 / degree-matched shuffled N')
display(summary.loc[focus, columns])

comparison = pd.read_csv(paths['arm_comparison_csv'])
focus_comparison = comparison.group_type.isin(['overall', 'fixed_clv_segment'])
print('2) 실제 N 조건부의 pooled·shuffled 대비 차이')
display(comparison.loc[focus_comparison])

with open(paths['json'], encoding='utf-8') as handle:
    payload = json.load(handle)
print('3) 사전 판정 규칙 결과')
print(json.dumps(payload['screen_reading'], ensure_ascii=False, indent=2))
print('결과 파일:', paths)